In [ ]:
import sys, os
sys.path.insert(0, '../utils')

In [ ]:
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.ticker import FormatStrFormatter, PercentFormatter, MaxNLocator
from utils import load_neurons_table
from activity_utils import require_activity_h5
from plot_utils import ex_color, inh_color
from figure7_utils import plot_inh_target_panel
from ensembles import require_ensemble_results, shared_input_by_size
import raster_utils


In [ ]:
# ── how this figure is drawn ─────────────────────────────────────────────────
# The panel drawing lives here rather than in utils/, so the notebook that produces
# figure 6 is readable end to end. utils/ensembles.py holds the analysis: detection,
# the matched controls and the bootstraps, all run once by scripts/ensemble_run.py.

import matplotlib
from matplotlib.colors import to_rgb
from matplotlib.patches import FancyArrowPatch
from matplotlib.legend_handler import HandlerPatch


CONTROL_COLOR  = "#999999"       # histogram fill (rendered with alpha=0.7)


CONTROL_FILL   = "#b8b8b8"       # solid fill for control triangles — pre-blended


                                 # 999999 @ alpha 0.7 on white so it matches the hist
SPINY_COLOR    = '#7C3AED'       # synapse onto a spine


SHAFT_COLOR    = '#059669'       # synapse onto shaft/soma


ex_color_edge      = "#8B0000"   # triangle outline, ensemble panels


control_color_edge = "#363535"   # triangle outline, control panels


def darken(color, factor=0.65):
    """A darker shade of `color` — for lines that must read against their own fill."""
    r, g, b = to_rgb(color)
    return (r * factor, g * factor, b * factor)


def star_below_legend(ax, star, fontsize=None, pad=0.02, color='black'):
    """Draw a significance star on its own line just under the legend box.

    Keeps the star out of the legend labels (where it crowds the value text)
    while still anchoring it to the legend rather than to a hand-tuned axes
    position: the star is left-aligned with the legend's label column and sits
    `pad` (axes fraction) below the bottom-most label, so it reads as one more
    line of the legend block regardless of how many entries there are.

    The anchor is the last label's text box, not `legend.get_window_extent()` —
    the latter includes the legend's border padding and handle column, which
    puts the star noticeably low and to the left of the labels.

    Must be called after `ax.legend(...)`. A legend computes its layout only
    while being drawn, so its artists have no meaningful extent before the first
    draw — hence the explicit `canvas.draw()` rather than reusing a renderer.
    """
    leg = ax.get_legend()
    if not star or leg is None:
        return None
    canvas = ax.get_figure().canvas
    canvas.draw()
    last = leg.get_texts()[-1]
    bb = last.get_window_extent(canvas.get_renderer()).transformed(ax.transAxes.inverted())
    if fontsize is None:
        fontsize = last.get_fontsize()
    return ax.text(bb.x0, bb.y0 - pad, star, transform=ax.transAxes,
                   ha='left', va='top', fontsize=fontsize, color=color)


def plot_null_hist_panel(ax, null_vals, obs_val, global_val, ctrl_color, obs_color,
                         obs_label, global_label, ctrl_label='Control',
                         bins=20, xlabel='', ylabel='Count', star=None,
                         legend_loc='upper left', legend_bbox=(0, 1.05),
                         headroom=1.5, marker_pad=1.03, ctrl_mean_color=None,
                         ctrl_mean_fmt='.3g', trim_yticks=True):
    """Null-distribution histogram + observed / control-mean vertical markers.

    Shared by the metric-B (connection probability) and metric-A (% synapses on
    spines) panels, which differ only in data, bins and x label.

    `ax.axvline` spans the full axes height, so with a `best`-placed legend the
    marker lines and the tallest bars run straight through the legend text. Here
    the markers are drawn with `vlines` capped just above the tallest bar
    (`marker_pad`) and the y limit is opened to `headroom` × that height, so the
    legend sits in a band that no artist reaches into.

    `star` (e.g. from `stats_corr.p_to_stars`) is drawn between the ensemble
    and control-mean vlines, including when it is 'ns'.

    The control-mean dashed line sits on top of the control bars, so drawing it
    in `ctrl_color` makes it near-invisible however opaque it is — same hue, and
    the bars behind it are only alpha-lightened. It is drawn in a darkened shade
    (`ctrl_mean_color`, default `darken(ctrl_color)`) so it reads as the same
    grey but stands off its own histogram.

    `trim_yticks` drops the y ticks that fall inside the headroom band. The band
    is deliberate — it is what keeps the legend off the bars — but no bar ever
    reaches it, so ticks up there label empty space. The limit is unchanged.

    Returns the histogram counts.
    """
    ctrl_mean = np.mean(null_vals)
    ctrl_leg_label = f'{ctrl_label} (mean = {ctrl_mean:{ctrl_mean_fmt}})'
    counts, _, _ = ax.hist(null_vals, bins=bins, color=ctrl_color, alpha=0.7,
                           label=ctrl_leg_label)
    y_top = counts.max() * marker_pad
    ax.vlines(obs_val, 0, y_top, color=obs_color, lw=2, label=obs_label)
    ax.vlines(global_val, 0, y_top, color='k', lw=1.5, ls='--', label=global_label)
    ax.vlines(ctrl_mean, 0, y_top, lw=2, ls='--',
              color=darken(ctrl_color) if ctrl_mean_color is None else ctrl_mean_color)
    ax.set_ylim(0, counts.max() * headroom)
    if trim_yticks:
        # set_yticks can rescale the view, so restore the limit afterwards —
        # the point is to lose the ticks, not the headroom they sat in.
        _ylim = ax.get_ylim()
        ax.set_yticks([t for t in ax.get_yticks() if 0 <= t <= counts.max()])
        ax.set_ylim(_ylim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    # Ensembles leads the legend. Draw order would put Control first — the
    # histogram has to be drawn before the vlines that cap to its height — but
    # ensembles is the result the panel is about, so the entries are reordered
    # rather than the drawing, which would change the z-order too.
    _h, _l = ax.get_legend_handles_labels()
    _by_label = dict(zip(_l, _h))
    _order = [lb for lb in (obs_label, ctrl_leg_label, global_label)
              if lb in _by_label]
    ax.legend([_by_label[lb] for lb in _order], _order,
              frameon=False, loc=legend_loc, bbox_to_anchor=legend_bbox)
    if star:
        x_lo = min(obs_val, ctrl_mean)
        x_hi = max(obs_val, ctrl_mean)
        x_mid = (x_lo + x_hi) / 2
        y_star = y_top
        ax.annotate('', xy=(x_hi, y_star), xytext=(x_lo, y_star),
                    arrowprops=dict(arrowstyle='-', color='black', lw=0.75, alpha=0.8),
                    annotation_clip=False)
        ax.text(x_mid, y_star, star, ha='center', va='bottom',
                fontsize=matplotlib.rcParams.get('font.size', 12), color='black',
                clip_on=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    return counts


def plot_shared_input_panel(ax, real_df, ctrl_df, ctrl_color, ens_color,
                            xlim=None, pct=95, floor=5,
                            inset_xlim=None, inset_pct=95,
                            inset_bounds=(0.55, 0.38, 0.43, 0.52),
                            inset_fontsize=8, star=None, inset_star=None,
                            legend_loc='upper left',
                            legend_bbox=(0.10, 1.04), verbose=True):
    """Shared-input histogram (I-only) with an inset for the E-only counts.

    Both distributions are drawn as unfilled `stairs` outlines with integer bins
    and `density=True`, so ensembles (n≈40) and control (n≈40k) are comparable.

    The main axes show `shared_inh`, the intersection of the members' inhibitory
    pre-synaptic sets — the flavour that carries the effect. The inset shows
    `shared_ex`, which lives on a much smaller range and would otherwise collapse
    into the first two bins of the main axes.

    Both x ranges are set the same way: the `pct` / `inset_pct` percentile of
    the *pooled* real+control values, rounded up and floored at `floor` so the
    axis never degenerates (the shared_ex p95 is 1 for this dataset). Because
    control dominates the pool (n≈40k vs 41), the cut follows the control
    distribution and clips the longer ensemble tail — the fraction of each
    group actually shown is logged. Pass `xlim` / `inset_xlim` to override.

    `star` / `inset_star` (e.g. from `stats_corr.p_to_stars`) annotate the two
    ensembles-vs-control tests, and are shown even when 'ns'. The main one is
    drawn under the legend by `star_below_legend`; the inset has no legend of
    its own, so its star goes in the inset's empty top-right corner.

    Returns (ax_inset, info_dict) where info_dict carries the max/percentile
    values that were logged.
    """
    ens_inh  = real_df['shared_inh'].values.astype(float)
    ctrl_inh = ctrl_df['shared_inh'].values.astype(float)
    ens_ex  = real_df['shared_ex'].values.astype(float)
    ctrl_ex = ctrl_df['shared_ex'].values.astype(float)

    def _stairs(axis, vals, color, label, xmax, lw):
        bins = np.arange(-0.5, xmax + 1.5, 1.0)
        counts, edges = np.histogram(vals, bins=bins, density=True)
        axis.stairs(counts, edges, color=color, linewidth=lw, label=label)

    def _pct_xlim(pooled, q, explicit):
        """Percentile-based integer x limit; `explicit` wins when given."""
        p = float(np.percentile(pooled, q))
        return (explicit if explicit is not None else int(max(floor, np.ceil(p)))), p

    # ── main axes: I-only shared input ──
    pooled_inh   = np.concatenate([ens_inh, ctrl_inh])
    xlim, p_inh  = _pct_xlim(pooled_inh, pct, xlim)
    for vals, color, label in [(ctrl_inh, ctrl_color, 'Control'),
                               (ens_inh,  ens_color,  'Ensembles')]:
        _stairs(ax, vals, color, label, xlim, 1.5)
    ax.set_xlim(-0.5, xlim + 0.5)
    ax.set_xticks(np.arange(0, xlim + 1, 5 if xlim > 15 else 2))
    ax.set_xlabel('Shared Inh input')
    ax.set_ylabel('Probability')
    # Legend nudged right of the x=0 spike so it clears both the tall first bin
    # and the inset, which occupies the upper-right quadrant.
    # Control is *drawn* first so the ensemble outline sits on top of it, but the
    # legend leads with Ensembles, matching the other panels' obs-then-null order.
    _h, _l = ax.get_legend_handles_labels()
    _order = [_l.index('Ensembles'), _l.index('Control')]
    ax.legend([_h[i] for i in _order], [_l[i] for i in _order],
              frameon=False, loc=legend_loc, bbox_to_anchor=legend_bbox)
    star_below_legend(ax, star)
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))

    # ── inset: E-only shared input ──
    pooled_ex            = np.concatenate([ens_ex, ctrl_ex])
    inset_xlim, p_ex     = _pct_xlim(pooled_ex, inset_pct, inset_xlim)

    ax_in = ax.inset_axes(inset_bounds)
    for vals, color in [(ctrl_ex, ctrl_color), (ens_ex, ens_color)]:
        _stairs(ax_in, vals, color, None, inset_xlim, 1.2)
    ax_in.set_xlim(-0.5, inset_xlim + 0.5)
    ax_in.set_xticks(np.arange(0, inset_xlim + 1))
    ax_in.set_xlabel('Shared Ex input', fontsize=inset_fontsize, labelpad=1)
    ax_in.set_ylabel('Probability', fontsize=inset_fontsize, labelpad=1)
    if inset_star:
        # The inset shares the main legend, so there is no legend box of its own
        # to sit under; its star goes in the inset's own top-right corner, which
        # the decaying distribution always leaves empty.
        ax_in.text(0.97, 0.95, inset_star, transform=ax_in.transAxes,
                   ha='right', va='top', fontsize=inset_fontsize + 2)
    ax_in.tick_params(labelsize=inset_fontsize - 1, length=2, pad=1)
    ax_in.spines[['top', 'right']].set_visible(False)
    ax_in.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    ax_in.patch.set_alpha(0)

    info = {'max_shared_ex_ens':  float(ens_ex.max()),
            'max_shared_ex_ctrl': float(ctrl_ex.max()),
            'max_shared_inh_ens':  float(ens_inh.max()),
            'max_shared_inh_ctrl': float(ctrl_inh.max()),
            f'p{pct}_shared_inh':      p_inh,
            f'p{inset_pct}_shared_ex': p_ex,
            'xlim': xlim, 'inset_xlim': inset_xlim}
    if verbose:
        # plain ASCII in the log lines: this also runs from consoles with a
        # non-UTF-8 codepage, where a literal arrow raises UnicodeEncodeError.
        for name, ens_v, ctrl_v, q, p, lim in [
            ('shared_inh', ens_inh, ctrl_inh, pct,       p_inh, xlim),
            ('shared_ex', ens_ex, ctrl_ex, inset_pct, p_ex, inset_xlim),
        ]:
            print(f"{name} max: ensembles={ens_v.max():.0f}  control={ctrl_v.max():.0f}  "
                  f"(pooled p{q}={p:.0f} -> x range 0-{lim}; shown: "
                  f"{(ens_v <= lim).mean():.1%} of ensembles, "
                  f"{(ctrl_v <= lim).mean():.1%} of control)")
    return ax_in, info


def plot_shared_input_by_size_panel(ax, d_size, ctrl_color, ens_color,
                                    metric='shared_inh', show_fold=True,
                                    fold_only_reliable=False,
                                    fold_fmt='{:.2f}×', fold_fontsize=8,
                                    show_inh_pct=False, inh_metrics=('shared_inh', 'shared_ex'),
                                    inh_fmt='{:.0f}% Inh',
                                    legend_loc='upper right', legend_bbox=None,
                                    bar_width=0.4,
                                    xlabel='Ensemble size (neurons)',
                                    ylabel='Shared I inputs per ensemble'):
    """Per-ensemble-size shared-input bars, in the figure-6 colour convention.

    Draws one measure onto an axes the caller already owns, in the red/grey
    ensemble/control convention, so it can sit inside a figure row.

    `d_size` is the output of `shared_input_by_size`; cumulative rows ('n>=3', 'n>=4')
    are dropped since they have no position on a size axis. Bar height is the
    **per-ensemble mean** (`value` / `value_null`), not the pooled sum, so bars
    stay comparable across sizes holding different numbers of ensembles.

    Note for the caption: shared input is an *intersection* over members, so it
    falls with ensemble size by construction — an interneuron must contact every
    member to count. The readable quantity is the ensemble/control gap, which is
    why the fold is annotated above each pair rather than left to the reader.

    That construction is also why `fold_only_reliable` exists. Once the ensembles
    are large enough, no interneuron reaches every member: both bars go to zero
    and the fold becomes 0/0.002 = '0.00x' printed above an invisible bar, which
    reads as a broken panel rather than as the real finding that the intersection
    is empty. `shared_input_by_size` already marks those rows `reliable=False` (null
    mean under its `min_null`); set this True to keep their bars but drop their
    annotations. Default False so existing callers are unchanged.

    `show_inh_pct` adds a second annotation line under the fold: what fraction of
    the shared input at that size is inhibitory, inh / (inh + ex) over the *observed*
    counts (`inh_metrics`, in that order). A presynaptic cell has exactly one
    clf_type, so the two counts partition the shared inputs and no null enters —
    this is the E/I composition table of ensembles.md §6.5, drawn in place. Sizes
    where the frame has no shared input at all (0/0) get the fold line only.

    Returns the per-size sub-frame actually plotted.
    """
    d = d_size[(d_size['metric'] == metric)
               & ~d_size['size'].astype(str).str.startswith('n>=')].copy()
    if d.empty:
        raise ValueError(f'no per-size rows for metric {metric!r} in d_size')
    d['size_i'] = d['size'].astype(int)
    d = d.sort_values('size_i')

    x = np.arange(len(d))
    ax.bar(x - bar_width / 2, d['value'], width=bar_width,
           color=ens_color, alpha=0.7, label='Ensembles')
    ax.bar(x + bar_width / 2, d['value_null'], width=bar_width,
           color=ctrl_color, alpha=0.8, label='Control')

    if show_fold:
        # Keyed on the raw `size` label so it lines up with `d` before the int cast.
        pct_by_size = {}
        if show_inh_pct:
            counts = {m: (d_size[d_size['metric'] == m]
                          .set_index(d_size.loc[d_size['metric'] == m, 'size']
                                     .astype(str))['obs_num'].to_dict())
                      for m in inh_metrics}
            for sz in d['size'].astype(str):
                inh, ex = (counts[m].get(sz, np.nan) for m in inh_metrics)
                if np.isfinite(inh) and np.isfinite(ex) and (inh + ex) > 0:
                    pct_by_size[sz] = 100.0 * inh / (inh + ex)

        # Annotated above the taller of the pair so the label never sits on a bar.
        # `reliable` may be absent from hand-built frames, hence the getattr.
        for xi, row in zip(x, d.itertuples()):
            if fold_only_reliable and not getattr(row, 'reliable', True):
                continue
            if np.isfinite(row.fold):
                txt = fold_fmt.format(row.fold)
                pct = pct_by_size.get(str(row.size))
                if pct is not None:
                    txt += '\n' + inh_fmt.format(pct)
                ax.annotate(txt, (xi, max(row.value, row.value_null)),
                            textcoords='offset points', xytext=(0, 2),
                            ha='center', va='bottom', fontsize=fold_fontsize)
        # Two annotation lines need roughly twice the headroom over the tallest bar.
        pad = 1.30 if pct_by_size else 1.18
        ax.set_ylim(0, max(d[['value', 'value_null']].to_numpy().max() * pad, 1e-9))

    ax.set_xticks(x)
    ax.set_xticklabels(d['size_i'])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    # legend_bbox nudges the legend off the tallest bar pair: at a narrow axes
    # width 'upper right' alone puts the swatches right against them.
    ax.legend(frameon=False, loc=legend_loc, bbox_to_anchor=legend_bbox)
    ax.spines[['top', 'right']].set_visible(False)
    return d


def _tri_marker_boundary_dist_pts(angle_rad, marker_area):
    """
    Points from a scatter '^' marker's origin (the point scatter places it at) to
    the triangle boundary along `angle_rad`. Marker vertices are at (0, R),
    (-R, -R), (R, -R) in points with R = 0.5 * sqrt(marker_area), matching
    matplotlib's `_set_triangle` scaling.

    Used to size FancyArrowPatch shrinks so every arrowhead lands the same small
    distance outside the target triangle regardless of the approach direction.
    """
    R = 0.5 * np.sqrt(marker_area)
    dx, dy = np.cos(angle_rad), np.sin(angle_rad)
    ts = []
    # Sides expressed as a*x + b*y = R for each of the 3 edges.
    for a, b in [(-2, 1), (0, -1), (2, 1)]:
        denom = a * dx + b * dy
        if abs(denom) > 1e-12:
            t = R / denom
            if t > 1e-6:
                ts.append(t)
    return min(ts) if ts else R


def plot_network_panel(ax, num_neurons=4, num_synapses=None, syn_color='black', 
                       spiny_syn=None, shaft_syn=None, neuron_color=ex_color, title="",
                       label_fontsize=13, marker_size=300, shrink=10,
                       edges=None, edge_colors=None, neuron_edge_color=None,
                       xlim=(0.0, 0.82), ylim=(0.12, 0.92), title_y=0.175,
                       node_coords=None):
    """
    Plots a single network panel with neurons and synaptic connections.
    """
    # Triangle outline follows the fill: dark red for ex, dark gray for control,
    # so ensemble and control panels stay distinguishable at figure scale.
    if neuron_edge_color is None:
        neuron_edge_color = control_color_edge if neuron_color == CONTROL_FILL else ex_color_edge
    # Fixed coordinates to mimic the image layout. Rows huddle closer vertically
    # so the schematic cluster occupies less of the ax without shrinking markers.
    # node_coords overrides these — the same 4 nodes, spread differently.
    # Order is fixed (0 top-left, 1 top-right, 2 bottom-left, 3 bottom-middle):
    # the edge tables below index into it, and node 2 is intentionally left
    # unconnected in the Ensemble panels.
    coords = np.array([
        [0.25, 0.60],  # 0: Top left
        [0.60, 0.60],  # 1: Top right
        [0.10, 0.38],  # 2: Bottom left
        [0.40, 0.38],  # 3: Bottom middle
    ]) if node_coords is None else np.asarray(node_coords, dtype=float)

    # No equal-aspect lock: scatter markers are drawn in point space, so the
    # triangles stay regular while the network fills the whole cell (less whitespace).
    ax.scatter(coords[:num_neurons, 0], coords[:num_neurons, 1], 
               marker='^', s=marker_size, color=neuron_color,
               edgecolor=neuron_edge_color, linewidth=0.8, zorder=2)
    
    # Edges mapped exactly to the schematic's layout.
    # First 3 form a V among nodes 0, 1, 3 (used in Ensemble panels) so node 2
    # is left without any connection — illustrates a non-participating neuron.
    potential_edges = [
        (0, 1), (1, 3), (0, 3),  # First 3 (used in Ensemble) — node 2 unconnected
        (3, 0), (2, 1), (2, 3)   # Extras
    ]
    
    edges_to_draw = []
    colors_to_draw = []
    
    # Explicit edges + colors take precedence — used for control panels
    # to draw a connectivity pattern different from the ensemble.
    if edges is not None:
        edges_to_draw = list(edges)
        if edge_colors is not None:
            colors_to_draw = list(edge_colors)
        else:
            colors_to_draw = [syn_color] * len(edges_to_draw)
    # Determine the number and colors of synapses based on the flags
    elif spiny_syn is not None or shaft_syn is not None:
        spiny = spiny_syn if spiny_syn else 0
        shaft = shaft_syn if shaft_syn else 0
        total_synapses = spiny + shaft
        
        edges_to_draw = potential_edges[:total_synapses]
        colors_to_draw = [SPINY_COLOR] * spiny + [SHAFT_COLOR] * shaft
        
    elif num_synapses is not None:
        if num_synapses == 1:
            # Use the specific diagonal connection for the random group
            edges_to_draw = [(2, 1)] 
        else:
            edges_to_draw = potential_edges[:num_synapses]
            
        colors_to_draw = [syn_color] * num_synapses

    # Tight bounds so the cluster fills the cell; set BEFORE drawing arrows so
    # transData reflects these limits when we probe direction for shrink calc.
    # Tighter limits zoom the node cluster up inside the axes — the nodes only
    # span x 0.10-0.60, y 0.38-0.60, so the defaults leave wide empty margins.
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    # Per-arrow shrinkA/shrinkB computed from the '^' marker geometry, so every
    # arrowhead sits the same small distance outside its target triangle
    # regardless of approach angle (side vs. vertex directions differ by ~2x).
    ALMOST_TOUCH_GAP_PTS = 1.5
    for (u, v), color in zip(edges_to_draw, colors_to_draw):
        p_u_disp = ax.transData.transform(coords[u])
        p_v_disp = ax.transData.transform(coords[v])
        dx_disp = p_v_disp[0] - p_u_disp[0]
        dy_disp = p_v_disp[1] - p_u_disp[1]
        if np.hypot(dx_disp, dy_disp) < 1e-9:
            continue
        angle_uv = np.arctan2(dy_disp, dx_disp)
        d_at_u = _tri_marker_boundary_dist_pts(angle_uv, marker_size)
        d_at_v = _tri_marker_boundary_dist_pts(angle_uv + np.pi, marker_size)
        arrow = FancyArrowPatch(
            posA=coords[u], posB=coords[v],
            arrowstyle='-|>', mutation_scale=16,
            color=color, linewidth=2.2,
            shrinkA=d_at_u + ALMOST_TOUCH_GAP_PTS,
            shrinkB=d_at_v + ALMOST_TOUCH_GAP_PTS,
            zorder=1
        )
        ax.add_patch(arrow)

    ax.axis('off')

    # Add the title label at the bottom using axes coordinates for pixel-perfect alignment
    if title:
        ax.text(0.0, title_y, title, transform=ax.transAxes,
                fontsize=label_fontsize, fontweight='normal', color='black')


def plot_network_panel_with_input(ax, label_fontsize=13, input_x=0.2, ens_x=0.8,
                                  show_internal_arrows=True,
                                  input_label_x=None, ens_label_x=None,
                                  ens_size=200, input_size=105,
                                  arrow_shrinkA=7, arrow_shrinkB=9,
                                  arrow_mutation=8):
    """
    Plots a deep learning style feedforward layout with an input column 
    and an ensemble column.

    `input_x` / `ens_x` are the two column positions in axes coordinates.
    Narrowing the gap (e.g. 0.30 / 0.74) shortens the all-to-all arrows and
    frees horizontal room — what the V3 row-D layout needs. Defaults are the
    original values, so V2 renders unchanged.
    """
    
    # Generate Y coordinates for evenly spaced nodes (top to bottom)
    # 7 Inputs (reversed so index 0 is at the top). Kept in the lower ~70% so the
    # top strip stays clear for the E/I legend.
    input_y = np.linspace(0.72, 0.06, 7) 
    
    # All shared inputs drawn as inhibitory circles
    input_inh_coords = np.column_stack((np.full(7, input_x), input_y))
    
    # 4 Ensemble neurons
    ens_y = np.linspace(0.60, 0.18, 4)
    ens_coords = np.column_stack((np.full(4, ens_x), ens_y))
    
    # ==========================================
    # Draw Nodes
    # ==========================================
    # ens_size / input_size are marker areas (pt^2), now params; arrow_shrinkB
    # is how far (pt) each arrowhead stops short of a triangle centre — keep it
    # near the triangle mid-line half-width (~0.4*sqrt(ens_size)) so heads touch.
    
    # Draw Input Neurons (7 Inhibitory circles)
    ax.scatter(input_inh_coords[:, 0], input_inh_coords[:, 1], 
               marker='o', s=input_size, color=inh_color,
               edgecolor='gray', linewidth=0.5, zorder=3)
               
    # Draw Ensemble Neurons (4 Excitatory triangles)
    ax.scatter(ens_coords[:, 0], ens_coords[:, 1], 
               marker='^', s=ens_size, color=ex_color,
               edgecolor=ex_color_edge, linewidth=0.8, zorder=3)
               
    # ==========================================
    # Draw Connections
    # ==========================================
    
    # 1. All-to-all: Inputs -> Ensemble (Thin gray arrows)
    all_inputs = input_inh_coords
    for inp in all_inputs:
        for ens in ens_coords:
            arrow = FancyArrowPatch(
                posA=inp, posB=ens,
                arrowstyle='-|>', mutation_scale=arrow_mutation,
                color='gray', linewidth=0.5, alpha=0.4, 
                shrinkA=arrow_shrinkA, shrinkB=arrow_shrinkB,
                zorder=1
            )
            ax.add_patch(arrow)
            
    # 2. Ensemble -> Ensemble spiny arrows. Span two rows apart so the bowed arc
    # has room to render (adjacent nodes are too close in the compressed layout).
    # show_internal_arrows=False drops them: these are E->E spine-targeting
    # arrows, which is Panel B/C's claim, not Panel D's. Panel D is about what
    # the ensemble receives, so the internal wiring is a distraction there.
    pairs = [(0, 2), (1, 3)] if show_internal_arrows else []
    for u, v in pairs:
        arrow = FancyArrowPatch(
            posA=ens_coords[u], posB=ens_coords[v],
            # Negative radius bows the arrows outside (to the right) into the empty space
            connectionstyle="arc3,rad=-0.5", 
            arrowstyle='-|>', mutation_scale=15,
            color=SPINY_COLOR, linewidth=2.2, 
            shrinkA=5, shrinkB=5,
            zorder=2
        )
        ax.add_patch(arrow)
        
    # Tight bounds so the schematic fills the cell
    ax.set_xlim(0.02, 0.98)
    ax.set_ylim(0.02, 0.98) 
    ax.axis('off')

    # ==========================================
    # Add Text Labels
    # ==========================================
    # Using ax.transData allows us to align text perfectly with the node X-coordinates
    # Label x defaults to the column x, but can be set wider than the columns
    # so the two captions clear each other when the columns are close.
    _in_lx  = input_x if input_label_x is None else input_label_x
    _ens_lx = ens_x   if ens_label_x   is None else ens_label_x
    ax.text(_in_lx, -0.06, "Shared input", transform=ax.transData,
            fontsize=label_fontsize, fontweight='normal', color='black', ha='center')

    ax.text(_ens_lx, -0.06, "Ensemble", transform=ax.transData,
            fontsize=label_fontsize, fontweight='normal', color='black', ha='center')


class _ArrowHandle:
    pass


class HandlerArrow(HandlerPatch):
    """Legend handler drawing a '-|>' arrow. `mutation_scale` is the head size
    and `lw` the shaft width, both in points — pass them up when the rest of the
    legend markers grow, or the arrow entry stays small next to them."""
    def __init__(self, mutation_scale=12, lw=1.5, color='black', **kwargs):
        super().__init__(**kwargs)
        self.mutation_scale = mutation_scale
        self.lw = lw
        self.color = color

    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):
        return [FancyArrowPatch(
            posA=(xdescent, ydescent + height / 2),
            posB=(xdescent + width, ydescent + height / 2),
            arrowstyle='-|>', mutation_scale=self.mutation_scale,
            color=self.color, lw=self.lw, transform=trans,
        )]

In [ ]:
# ── what this figure needs, checked up front ─────────────────────────────────
# Neither of the two things below is in the Zenodo snapshot. Both are produced once,
# locally, in this order — each guard prints exactly how if it is not satisfied.
#
#   1. the two MICrONS activity H5 files
#          scripts/extract_calcium_data_via_docker.ipynb, run inside the MICrONS
#          database container (hours; ~19 GB out)
#   2. the ensemble run pickles, which read them
#          python scripts/ensemble_run.py                (1–3 hours)
#
# Panel A reads the H5 files directly (it re-runs detection on one scan); panels B–E
# read only the pickle. To render B–E without the recordings, comment out the first
# guard and skip the panel-A cell below.
require_activity_h5(needed_by='figure 6 panel A')
lds_results_path = require_ensemble_results('lds')

In [ ]:
with open(lds_results_path, 'rb') as f:
    results = pickle.load(f)

all_ensembles_df          = results['all_ensembles_df']
ensemble_members_by_scank = results['ensemble_members_by_scank']
matched_controls_df          = results['matched_controls_df']
spine_targeting            = results['spine_targeting']
connection_probability            = results['connection_probability']
shared_input            = results['shared_input']
print(f'Loaded {len(all_ensembles_df)} ensembles across '
      f'{all_ensembles_df[["session", "scan_idx"]].drop_duplicates().shape[0]} scans')

In [ ]:
# shared_input_by_size runs a 1000-replicate bootstrap for every metric, so it is
# cached across re-runs of this cell - `del d_size_v3` to force a recompute.
try:
    d_size_v3
except NameError:
    d_size_v3 = shared_input_by_size(results)

# Column-wide baselines, independent of which detector ran.
# Panel E: global I->E spine fraction, 16,668 / 72,207 tagged I->E synapses.
P_GLOBAL_IE_SPINE = 16668 / 72207
# Panel C: 29,735 / 36,176 over ALL E->E synapses, not the tagged-only denominator.
_p_global_c3 = 29735 / 36176
# Panel B: over all ordered pairs of the 1,188 column excitatory neurons.
global_EE_conn_prob = 36176 / (1188 * 1187)
print(f'Global I->E spine fraction:        {P_GLOBAL_IE_SPINE:.4f}')
print(f'Global E->E spine fraction:        {_p_global_c3:.4f}')
print(f'Global E->E connection probability: {global_EE_conn_prob:.6f}')


In [ ]:
# control font size
plt.rcParams['font.size'] = 19
plt.rcParams['legend.fontsize'] = 15
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.labelsize'] = 15
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.labelsize'] = 15
plt.rcParams['font.family'] = 'Arial'

LABEL_FONTSIZE   = 14   # schematic captions ('Ensemble' / 'Control' / 'Shared input' / …)
LEG_FONTSIZE     = 14   # inner legends in B / C / D
letter_font_size = 24   # 'A' 'B' 'C' 'D' 'E' panel letters
INSET_FONTSIZE   = 13   # Panel D — shared-E-input inset labels/ticks/star
FOLD_FONTSIZE    = 11.5   # Panel D — 'N.NNx' fold annotations above by-size bars
RASTER_FONTSIZE  = 14   # Panel A — every raster text (row labels, clip nums, scale bar, …)


In [ ]:
# Panel A re-runs detection on one scan straight from the calcium H5 - the only
# part of this figure that needs it. Panels B-E come from the pickle above.
_neurons_df = load_neurons_table(use_column_manual_ct=True)

_RASTER_SESSION  = 6
_RASTER_SCAN_IDX = 4

_ens_result, _ens_rids, _Z, _pool_units = raster_utils.detect_scan_ensembles(
    _neurons_df, _RASTER_SESSION, _RASTER_SCAN_IDX)
_ranked      = raster_utils.rank_ensembles(_ens_result, _Z)
_k           = _ranked[0]   # top-ranked ensemble, by activation energy
_all_members = raster_utils.ensemble_members(_ens_result, _ens_rids, _k)

# Display-only trim (MAX_MEMBERS, MEMBER_PICK_SEED): a random subset of the
# full membership, so a busy ensemble still fits the panel legibly.
_member_rids = _all_members
if raster_utils.MAX_MEMBERS and len(_all_members) > raster_utils.MAX_MEMBERS:
    _rng  = np.random.default_rng(raster_utils.MEMBER_PICK_SEED)
    _keep = sorted(_rng.choice(len(_all_members), size=raster_utils.MAX_MEMBERS,
                               replace=False))
    _member_rids = [_all_members[i] for i in _keep]

_pool_by_rid   = {r: p for r, p in _pool_units}
_panel_a_units = [(r, _pool_by_rid[r]) for r in _member_rids if r in _pool_by_rid]

_oracle_windows = raster_utils.load_oracle_windows(_RASTER_SESSION, _RASTER_SCAN_IDX)
panel_a_rasters = raster_utils.build_oracle_rasters(_panel_a_units, _oracle_windows)
panel_a_fps     = float(_panel_a_units[0][1]['fps'])
panel_a_rids    = [r for r, _p in _panel_a_units]

_ct_dict = _neurons_df.set_index('root_id')['cell_type'].to_dict()
panel_a_cell_types = [_ct_dict.get(int(r), '?') for r in panel_a_rids]

print(f's{_RASTER_SESSION} sc{_RASTER_SCAN_IDX}: k={_k} '
      f'({len(_member_rids)} of {len(_all_members)} members)  fps={panel_a_fps:.3f}')
print(f'Loaded root_ids: {raster_utils.neuron_labels(panel_a_rids, _neurons_df)}')


In [ ]:
# =============================================================================
# LAYOUT CONFIG — every size/spacing knob for this figure lives here
# =============================================================================
# Grid
FIG_SIZE      = (20.5, 14.0) # whole figure, inches. Widened from 19 purely
                             #   to feed panel D's by-size bars; every other
                             #   panel keeps its absolute width and only gains.
ROW_HEIGHTS   = [1.25, 1.1, 1.10]  # relative heights of rows A / B-C / D-E
ROW_GAP       = 0.02         # vertical gap between the three rows

# Row 1 — A
A_RASTER_HIST = [3.0, 1.0]   # raster : ensemble-size histogram, left→right
A_GAP         = 0.06
A_RASTER_MARG = dict(left_in=0.55, right_in=0.4, bottom_in=0.7)
                             # fixed-inch margins *inside* the raster subfigure.
                             # Raising right_in shrinks the clip grid without
                             # giving the histogram more width — use it to pull
                             # the grid back toward V2's proportions.

# Custom legends inside the B / C / D schematics
LEG_MARKER    = 13           # triangle & dot size in those legends (points)

LEG_ARROW     = 18           # arrowhead size of B's 'Synapse' entry

# Row 2 — B | C
BC_GAP        = 0.05         # gap between the B block and the C block
BC_SCH_HIST   = [3.3, 2.0]   # inside B (and C): schematics : histogram.
                             #   Raise the first number to grow the drawings;
                             #   the histogram then matches E's histogram width.
BC_MARKER     = 400          # triangle size (marker *area*, pt^2)
BC_NODES      = [[0.25, 0.60],   # the 4 triangles (0 top-L, 1 top-R,
                 [0.55, 0.60],   #   2 bottom-L, 3 bottom-mid). Spread wider in
                 [0.10, 0.35],   #   y than the old 0.38-0.60 so the cluster is
                 [0.40, 0.35]]   #   tall and narrow — that vertical stretch is
                             #   what pulls it away from the histogram. Keep the
                             #   column order; the edges index into it.
BC_XLIM       = (0.04, 0.66) # data window of the node cluster: tighten to zoom
BC_YLIM       = (0.14, 0.84) #   in (less white space), widen to zoom out.
BC_BOX_ASPECT = ((BC_YLIM[1] - BC_YLIM[0]) / (BC_XLIM[1] - BC_XLIM[0]))
                             # schematic axes height / width. Tied to the window
                             # so one data unit is the same length in x and y —
                             # any other value shears the triangle cluster.
BC_TITLE_Y    = 0.06         # 'Ensemble'/'Control' height, axes fraction —
                             #   lowered to clear the now-lower bottom nodes
BC_HIST_MARG  = dict(left=0.15, right=0.97, top=0.88, bottom=0.17)

# Row 3 — D | E
DE_SPLIT      = [1.88, 1.18] # width of the D block : the E block. ~1.1 in has
                             #   moved from E to D across three passes. It costs
                             #   E nothing drawn: its two schematics are aspect-
                             #   locked and height-limited, so their box was
                             #   mostly padding, and E_SCH_HIST below keeps the
                             #   histogram's width while the padding goes.
DE_GAP        = 0      # bare strip between the two blocks. What is left
                             #   between the by-size axes and panel E is 0.33 in
                             #   to E's first ink — which is the 'Shared input'
                             #   caption, overhanging its circles by 0.7 in at the
                             #   bottom. To the circles themselves it is ~1.05 in,
                             #   so the strip still looks emptier than it is.

# Panel D
D_NET_PLOTS   = [0.95, 3.76] # input schematic : the two plots beside it. First
                             #   value down (was 1.15) hands width to the plots;
                             #   the columns move closer so the network still
                             #   fits its narrower subfigure.
D_PLOTS_SPLIT = [1.55, 1.0]  # shared-input histogram : by-size bars — raise
                             #   the first value to widen the shared-I histogram
                             #   and push the by-size bars further right.
                             #   Was 3.1 when the by-size bars carried one
                             #   annotation line. 'N.NNx' over 'NN% Inh' is
                             #   ~0.65 in wide at FOLD_FONTSIZE, so the five bar
                             #   groups need >=3.65 in of axes before two
                             #   neighbouring labels overlap; 1.02 gives them
                             #   4.28 in, which also fills the strip that used to
                             #   sit between this panel and E. The width came from
                             #   D_PLOTS_WSPACE, DE_SPLIT and the wider figure,
                             #   plus 0.4 in off the shared-I histogram (4.77 ->
                             #   4.37 in — still the widest axes in the row).
D_NET_ASPECT  = 1.30         # schematic axes height / width; taller now that
                             #   the two columns sit closer horizontally
D_INPUT_X     = 0.34         # x of the inhibitory column, axes fraction
D_ENS_X       = 0.60         #   and of the ensemble column. Closer = shorter
                             #   arrows. The captions no longer limit this — they
                             #   have their own x below — so pull as tight as the
                             #   arrows still read.
D_INLABEL_X   = 0.22         # 'Shared input' / 'Ensemble' caption centres. Set
D_ENSLABEL_X  = 0.74         #   wider than the columns so the two captions clear
                             #   each other while the columns stay close.
D_ENS_SIZE    = 150          # ex-triangle marker area (pt^2); smaller than the
                             #   old 200 so the arrows aren't dwarfed by them
D_ARROW_SHRB  = 4            # how far (pt) each arrowhead stops short of a
                             #   triangle. Lower = arrows reach further in. ~0.4*
                             #   sqrt(D_ENS_SIZE) lands them on the neuron.
D_ALL_LEG_BB  = (0.03, 1.04) # shared-input histogram legend anchor; lower the
                             #   first value to move it left (the function's
                             #   default is 0.10, clearing the x=0 spike)
D_SIZE_LEG_BB = (1.00, 1.02) # by-size legend anchor. It used to sit outside the
                             #   axes (1.65) because the axes was too narrow to
                             #   hold it; at 3.2 in it fits inside, above the
                             #   short n=4..6 bars, and stops crowding panel E.
D_PLOTS_WSPACE = 0.18        # gap between D's two plots, in mean-axes-width
                             #   units. 0.34 left ~1.1 in of white space there;
                             #   0.18 is ~0.65 in, still clear of the by-size
                             #   y label and its ticks.

# Panel E
E_SCH_HIST    = [2.30, 2.0] # the two cell schematics : histogram. Lowered twice
                             #   from 2.7, each time paired with a DE_SPLIT move so
                             #   the histogram keeps its width (3.15 in) and only
                             #   the schematics' padding is spent. Their axes is
                             #   now 2.05 in against 2.11 in of drawn content: the
                             #   drawing is aspect-locked and height-limited, so it
                             #   does not shrink — it hangs ~0.03 in over each side,
                             #   which is why clip_on is off on *both* schematics
                             #   below.
E_SCALE       = 0.65         # pyramidal cell size
E_PITCH       = 1.80         # vertical spacing between cells. Must stay above
                             #   2.5 x E_SCALE or one trunk spears the soma above
E_INH_SIZE    = 180          # inhibitory circle size (marker area, pt^2)
E_SOMA        = dict(soma_w=0.35, soma_top=0.30, soma_bot=0.42)
                             # soma triangle only (half-width, apex, base), in
                             # cell-local units — defaults are 0.50/0.40/0.55.
                             # trunk_len and spine_len are deliberately not here:
                             # shrinking the soma leaves the dendrite and its
                             # spines exactly as they are.
E_EX_X        = [1.3, 1.05, 1.17]   # x of each cell; lower = closer to circles
E_ENS_DX      = 0.45         # shift the whole ensemble schematic (its 3 circles
                             #   AND 3 cells) right by this much, so all 6 sit
                             #   near the divider, close to the control panel's 6.
                             #   Control panel is left unshifted.
E_XLIM        = (-0.70, 1.85)        # keep the right edge just past the cells:
                                     #   the axes is aspect-locked, so unused
                                     #   x-range only shrinks the drawing
E_CAPTION_Y   = 0.005        # 'Shared input'/'Ensemble'/'Control' height,
                             #   axes fraction — lower drops them further down
E_HIST_MARG   = dict(left=0.13, right=0.99, top=0.90, bottom=0.17)
                             # left/right trimmed from 0.17/0.97: those margins
                             # were the last free width in the E block. 0.13 x the
                             # subfigure is still ~0.5 in, which holds 'Count' plus
                             # 3-digit ticks.
# =============================================================================

fig3 = plt.figure(figsize=FIG_SIZE, dpi=600)
sfs3 = fig3.subfigures(3, 1, height_ratios=ROW_HEIGHTS, hspace=ROW_GAP)

# =========================================================================
# ROW 1 — A: oracle raster (wide) + ensemble-size histogram
# =========================================================================
# width_ratios pushed to 3:1 (V2 used 2.25:1) — the raster is the panel that
# needs the width; the size histogram reads fine small.
sf3_a_raster, sf3_a_hist = sfs3[0].subfigures(1, 2, width_ratios=A_RASTER_HIST,
                                              wspace=A_GAP)
raster_utils.plot_oracle_rasters(
    panel_a_rasters, fps=panel_a_fps, fig=sf3_a_raster,
    clips_group_title='Oracle clips', row_labels=panel_a_cell_types,
    show_time_label=False, fontsize=RASTER_FONTSIZE, **A_RASTER_MARG)

ax3_a_hist = sf3_a_hist.subplots(1, 1)
sf3_a_hist.subplots_adjust(left=0.18, right=0.95, top=0.86, bottom=0.20)
sns.histplot(data=all_ensembles_df, x='n_members', bins=30,
             color=ex_color, alpha=0.7, discrete=True, ax=ax3_a_hist)
ax3_a_hist.set_xlabel('Ensemble size (neurons)')
ax3_a_hist.set_ylabel('Count')
ax3_a_hist.spines[['top', 'right']].set_visible(False)
_xt3 = np.sort(all_ensembles_df['n_members'].unique())
if len(_xt3) < 15:
    ax3_a_hist.set_xticks(_xt3)
    ax3_a_hist.set_xticklabels(_xt3)
ax3_a_hist.yaxis.set_major_locator(MaxNLocator(nbins=4, integer=True))
ax3_a_hist.yaxis.set_major_formatter(FormatStrFormatter('%g'))

sfs3[0].text(0.005, 0.97, 'A', fontsize=letter_font_size, fontweight='bold', va='top')

# =========================================================================
# ROW 2 — B: connection probability  |  C: internal spine fraction
# =========================================================================
# B left / C right = reading order; swap the two subfigure blocks to flip.
# Two things keep the schematics from looking stretched and lost in white space:
# an explicit box_aspect (plot_network_panel places its nodes in axes fractions,
# so a tall thin axes pulls the four triangles apart and lengthens every arrow),
# and tightened x/y limits — the node cluster only spans x 0.10-0.60, y 0.38-0.60,
# so the function's default window leaves wide empty margins around it.
_bc_panel_kw = dict(label_fontsize=LABEL_FONTSIZE, marker_size=BC_MARKER,
                    xlim=BC_XLIM, ylim=BC_YLIM, title_y=BC_TITLE_Y,
                    node_coords=BC_NODES)
sf3_b, sf3_c = sfs3[1].subfigures(1, 2, width_ratios=[1, 1], wspace=BC_GAP)

# ---- B ----
sf3_b_sch, sf3_b_hist = sf3_b.subfigures(1, 2, width_ratios=BC_SCH_HIST, wspace=0.0)
ax3_b_ens, ax3_b_rand = sf3_b_sch.subplots(1, 2)
sf3_b_sch.subplots_adjust(wspace=0.04, left=0.02, right=0.99, top=0.97, bottom=0.03)
for _ax in (ax3_b_ens, ax3_b_rand):
    _ax.set_box_aspect(BC_BOX_ASPECT)
ax3_b = sf3_b_hist.subplots(1, 1)
sf3_b_hist.subplots_adjust(**BC_HIST_MARG)

plot_network_panel(ax3_b_ens, num_synapses=3, syn_color='black',
                   title='Ensemble', **_bc_panel_kw)
plot_network_panel(ax3_b_rand, edges=[(3, 0)], syn_color='black',
                   title='Control', neuron_color=CONTROL_FILL, **_bc_panel_kw)

_ex_tri3 = mlines.Line2D([], [], color='w', marker='^', markerfacecolor='none',
                         markeredgecolor='black', markeredgewidth=1.2,
                         markersize=LEG_MARKER, label='Excitatory')
ax3_b_ens.legend(handles=[_ex_tri3, _ArrowHandle()],
                 labels=['Excitatory', 'Synapse'],
                 handler_map={_ArrowHandle: HandlerArrow(mutation_scale=LEG_ARROW,
                                                         lw=2.0)},
                 frameon=False, loc='upper left', bbox_to_anchor=(0.0, 1.02),
                 fontsize=LEG_FONTSIZE)

plot_null_hist_panel(
    ax3_b, connection_probability['valid_null'],
    obs_val=connection_probability['p_obs'], global_val=global_EE_conn_prob,
    ctrl_color=CONTROL_COLOR, obs_color=ex_color,
    obs_label=f'Ensembles ({connection_probability["p_obs"]:.1%})',
    global_label=f'Baseline ({global_EE_conn_prob:.1%})', headroom=1.5, legend_bbox=(0, 1.125),
    ctrl_mean_fmt='.0%', star=connection_probability['star'],
    bins=20, xlabel='Connection probability (%)')
ax3_b.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))

sf3_b.text(0.01, 0.96, 'B', fontsize=letter_font_size, fontweight='bold', va='top')

# ---- C ----
sf3_c_sch, sf3_c_hist = sf3_c.subfigures(1, 2, width_ratios=BC_SCH_HIST, wspace=0.0)
ax3_c_ens, ax3_c_rand = sf3_c_sch.subplots(1, 2)
sf3_c_sch.subplots_adjust(wspace=0.04, left=0.02, right=0.99, top=0.97, bottom=0.03)
for _ax in (ax3_c_ens, ax3_c_rand):
    _ax.set_box_aspect(BC_BOX_ASPECT)
ax3_c = sf3_c_hist.subplots(1, 1)
sf3_c_hist.subplots_adjust(**BC_HIST_MARG)

plot_network_panel(ax3_c_ens, spiny_syn=3, shaft_syn=0,
                   title='Ensemble', **_bc_panel_kw)
plot_network_panel(ax3_c_rand, edges=[(1, 0), (3, 1), (0, 3)],
                   edge_colors=[SPINY_COLOR, SPINY_COLOR, SHAFT_COLOR],
                   title='Control', neuron_color=CONTROL_FILL, **_bc_panel_kw)

_purple_arrow3 = _ArrowHandle()
_green_arrow3 = _ArrowHandle()
ax3_c_ens.legend(
    handles=[_purple_arrow3, _green_arrow3],
    labels=['Onto spine', 'Onto shaft/soma'],
    handler_map={
        _purple_arrow3: HandlerArrow(mutation_scale=LEG_ARROW, lw=2.0,
                                     color=SPINY_COLOR),
        _green_arrow3: HandlerArrow(mutation_scale=LEG_ARROW, lw=2.0,
                                    color=SHAFT_COLOR),
    },
    frameon=False, loc='upper left', bbox_to_anchor=(0.0, 1.04),
    fontsize=LEG_FONTSIZE)


plot_null_hist_panel(
    ax3_c, spine_targeting['valid_null'],
    obs_val=spine_targeting['p_obs'], global_val=_p_global_c3,
    ctrl_color=CONTROL_COLOR, obs_color=ex_color,
    obs_label=f'Ensembles ({spine_targeting["p_obs"]:.0%})',
    global_label=f'Baseline ({_p_global_c3:.0%})',
    ctrl_mean_fmt='.0%', star=spine_targeting['star'], headroom=1.5, legend_bbox=(0, 1.125),
    bins=30, xlabel='% of synapses on spines')
ax3_c.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
# Auto ticks land on 60/80 only; 100% is the natural right anchor for a
# percentage axis, so pin the three and open the limit far enough to show it.
ax3_c.set_xticks([0.6, 0.8, 1.0])
ax3_c.set_xlim(right=max(ax3_c.get_xlim()[1], 1.0))

sf3_c.text(-0.025, 0.96, 'C', fontsize=letter_font_size, fontweight='bold', va='top')

# =========================================================================
# ROW 3 — D: shared I input  |  E: spine fraction of the shared I input
# =========================================================================
sf3_d, sf3_e = sfs3[2].subfigures(1, 2, width_ratios=DE_SPLIT, wspace=DE_GAP)

# ---- D ----
# Same nesting as row 2: the input schematic gets its own subfigure, the two
# plots keep normal axes margins. The by-size bars are the smallest thing here
# on purpose — 5 bar pairs need far less width than the 26-bin histogram beside
# them, and shrinking them is what uncrowds the row.
sf3_d_net, sf3_d_plots = sf3_d.subfigures(1, 2, width_ratios=D_NET_PLOTS, wspace=0.0)
ax3_d_net = sf3_d_net.subplots(1, 1)
sf3_d_net.subplots_adjust(left=0.02, right=0.99, top=0.97, bottom=0.03)
ax3_d_net.set_box_aspect(D_NET_ASPECT)
ax3_d_all, ax3_d_size = sf3_d_plots.subplots(1, 2, width_ratios=D_PLOTS_SPLIT)
sf3_d_plots.subplots_adjust(wspace=D_PLOTS_WSPACE, left=0.06, right=1.0,
                            top=0.90, bottom=0.17)

# Columns pulled together (D_INPUT_X / D_ENS_X vs the 0.2/0.8 default) to free
# width for the two histograms that share this row.
plot_network_panel_with_input(ax3_d_net, label_fontsize=LABEL_FONTSIZE,
                              input_x=D_INPUT_X, ens_x=D_ENS_X,
                              input_label_x=D_INLABEL_X, ens_label_x=D_ENSLABEL_X,
                              ens_size=D_ENS_SIZE, arrow_shrinkB=D_ARROW_SHRB,
                            show_internal_arrows=False)

_ex_h3 = mlines.Line2D([], [], color='w', marker='^', markerfacecolor=ex_color,
                       markersize=LEG_MARKER, label='Excitatory',
                       markeredgecolor=ex_color_edge, markeredgewidth=0.8)
_inh_h3 = mlines.Line2D([], [], color='w', marker='o', markerfacecolor=inh_color,
                        markersize=LEG_MARKER, label='Inhibitory',
                        markeredgecolor='gray', markeredgewidth=0.5)
ax3_d_net.legend(handles=[_ex_h3, _inh_h3], frameon=False, loc='upper left',
                 bbox_to_anchor=(0.10, 1.18), fontsize=LEG_FONTSIZE)

ax3_d_ee, _shared_info3 = plot_shared_input_panel(
    ax3_d_all, shared_input['real_df'], shared_input['ctrl_df'],
    ctrl_color=CONTROL_COLOR, ens_color=ex_color, inset_pct=99, pct=99,
    inset_fontsize=INSET_FONTSIZE, legend_bbox=D_ALL_LEG_BB,
    star=None, inset_star=None, verbose=False)
ax3_d_all.set_yticks([0, 0.1, 0.2, 0.3])
ax3_d_all.set_ylim(0, 0.31)

# Shared input is an intersection over members, so it falls with ensemble size
# by construction — an interneuron has to contact every member to count. The
# fold annotations are what carry the content; say so in the caption.
plot_shared_input_by_size_panel(
    ax3_d_size, d_size_v3, ctrl_color=CONTROL_COLOR, ens_color=ex_color,
    metric='shared_inh', ylabel='Mean shared Inh inputs',
    show_inh_pct=True,
    fold_fontsize=FOLD_FONTSIZE, legend_bbox=D_SIZE_LEG_BB)
ax3_d_size.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax3_d_size.yaxis.set_major_formatter(FormatStrFormatter('%g'))

sf3_d.text(0.008, 0.97, 'D', fontsize=letter_font_size, fontweight='bold', va='top')

# ---- E ----
# The pyramidal schematics carry the most drawn detail in the figure (spine
# heads on each dendrite), so they get the largest share here and run to nearly
# the full subfigure height.
sf3_e_sch, sf3_e_hist = sf3_e.subfigures(1, 2, width_ratios=E_SCH_HIST, wspace=0.0)
ax3_e_ens, ax3_e_ctrl = sf3_e_sch.subplots(1, 2)
sf3_e_sch.subplots_adjust(wspace=0.02, left=0.01, right=0.99, top=0.99, bottom=0.02)
# Ensemble drawn on top of control; its axes patch stays transparent so lifting
# it doesn't paint over the neighbour, only wins where artists overlap.
ax3_e_ens.set_zorder(ax3_e_ctrl.get_zorder() + 1)
ax3_e_ens.patch.set_visible(False)
ax3_e_hist = sf3_e_hist.subplots(1, 1)
sf3_e_hist.subplots_adjust(**E_HIST_MARG)

# Row 3 is a busy row, so the schematic is packed tighter than the single-panel
# version: smaller inhibitory markers and the pyramidal column pulled left
# toward them (E_INH_SIZE / E_EX_X).
_e_ys = np.array([2, 1, 0]) * E_PITCH + 0.15
_inh_kw3 = dict(
    inh_color=inh_color, spiny_color=SPINY_COLOR, shaft_color=SHAFT_COLOR,
    spine_rank_by_target=(0, 1, 2),
    spiny_lw=1.6, spiny_alpha=1.0, spiny_head=14,
    shaft_lw=1.0, shaft_alpha=1, shaft_head=9,
    n_soma_arrows=1, label_fontsize=LABEL_FONTSIZE, caption_y=E_CAPTION_Y,
    neuron_scale=E_SCALE, inh_size=E_INH_SIZE, geom_kw=E_SOMA,
    xlim=E_XLIM,
    ylim=(-0.55, _e_ys[0] + 1.95 * E_SCALE + 0.30),
)
# Base coords (control panel). The ensemble panel adds E_ENS_DX to every x.
_e_inh_xy = [[0.0, _y + 0.25] for _y in _e_ys]
_e_ex_xy  = [[_x, _y] for _x, _y in zip(E_EX_X, _e_ys)]
_e_inh_xy_ens = [[x + E_ENS_DX, y] for x, y in _e_inh_xy]
_e_ex_xy_ens  = [[x + E_ENS_DX, y] for x, y in _e_ex_xy]
plot_inh_target_panel(ax3_e_ens, n_spiny=3, soma_color=ex_color,
                      dendrite_color=ex_color_edge, title='Ensemble',
                      input_title='Shared input',
                      title_dx=10, input_title_dx=-10,
                      inh_xy=_e_inh_xy_ens, ex_xy=_e_ex_xy_ens, **_inh_kw3)
# The rightward E_ENS_DX shift pushes the right soma/spines a touch past the axes
# right edge; without this they clip to the bbox and look trimmed. Clipping off
# (+ the higher zorder set above) lets the overflow draw over the seam.
for _art in (*ax3_e_ens.patches, *ax3_e_ens.lines,
             *ax3_e_ens.collections, *ax3_e_ens.texts):
    _art.set_clip_on(False)
# input_title='' on the control panel: the inhibitory column is the same
# population in both, so repeating the label only adds ink and is what collides
# with 'Control' at this width.
plot_inh_target_panel(ax3_e_ctrl, n_spiny=1, soma_color=CONTROL_FILL,
                      dendrite_color=control_color_edge, title='Control',
                      input_title='',
                      inh_xy=_e_inh_xy, ex_xy=_e_ex_xy, **_inh_kw3)
# The control panel needs the same treatment now: its axes box is 2.05 in against
# 2.11 in of drawing, so ~0.03 in hangs over each side and clipping would shave
# the outer spine tips. Both neighbours there are white space.
for _art in (*ax3_e_ctrl.patches, *ax3_e_ctrl.lines,
             *ax3_e_ctrl.collections, *ax3_e_ctrl.texts):
    _art.set_clip_on(False)

_sp_ii = shared_input['shared_inh_spine_frac']
plot_null_hist_panel(
    ax3_e_hist, _sp_ii['null'],
    obs_val=_sp_ii['obs'], global_val=P_GLOBAL_IE_SPINE,
    ctrl_color=CONTROL_COLOR, obs_color=ex_color,
    obs_label=f'Ensembles ({_sp_ii["obs"]:.1%})',
    global_label=f'Baseline ({P_GLOBAL_IE_SPINE:.1%})',
    ctrl_mean_fmt='.0%', star=_sp_ii['star'], headroom=1.5, legend_bbox=(0, 1.125),
    bins=30, xlabel='% of shared Inh synapses on spines')
ax3_e_hist.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))

print(f'Panel E  obs={_sp_ii["obs"]:.4f}  null={np.mean(_sp_ii["null"]):.4f}  '
      f'p={_sp_ii["p_emp"]:.4f} {_sp_ii["star"]}  '
      f'n={_sp_ii["n_num"]}/{_sp_ii["n_den"]}  '
      f'(global I→E = {P_GLOBAL_IE_SPINE:.4f})')

sf3_e.text(0.075, 0.97, 'E', fontsize=letter_font_size, fontweight='bold', va='top')

plt.savefig('fig6.pdf', format='pdf', bbox_inches='tight')
